# Lab 3 Advanced Analytics

Ecommerce Transactions Analysis

In [ ]:

# Cell 1: Imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import shapiro, zscore
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)


In [ ]:

# Cell 2: Load Dataset
df = pd.read_csv('ecommerce_dataset.csv')
print(df.shape)
df.head()


In [ ]:

# Cell 3: Data Quality Assessment

missing_count = df.isnull().sum()
missing_percent = df.isnull().mean()*100

data_quality_report = pd.DataFrame({
    'Missing_Count': missing_count,
    'Missing_Percentage': missing_percent
})

duplicate_rows = df.duplicated().sum()

print("Duplicate Rows:", duplicate_rows)
print(data_quality_report)

data_quality_report.to_csv('data_quality_report.csv')


In [ ]:

# Cell 4: Statistical Profiling

numeric_cols = df.select_dtypes(include=np.number).columns

stats_profile = df[numeric_cols].describe().T
stats_profile['median'] = df[numeric_cols].median()
stats_profile['skewness'] = df[numeric_cols].skew()
stats_profile['kurtosis'] = df[numeric_cols].kurt()

stats_profile.to_csv('statistical_profile.csv')
stats_profile.head()


In [ ]:

# Cell 5: Normality Tests

results = []

for col in numeric_cols[:3]:
    sample = df[col].dropna()
    if len(sample) > 5000:
        sample = sample.sample(5000, random_state=42)

    stat,p = shapiro(sample)
    results.append([col, stat, p])

normality_df = pd.DataFrame(results,
                            columns=['Variable','Statistic','P_Value'])

normality_df


In [ ]:

# Cell 6: Outlier Detection

outlier_summary = []

for col in numeric_cols:

    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5*iqr
    upper = q3 + 1.5*iqr

    iqr_count = ((df[col] < lower) | (df[col] > upper)).sum()

    z_count = (np.abs(zscore(df[col], nan_policy='omit')) > 3).sum()

    outlier_summary.append([col, iqr_count, z_count])

outlier_df = pd.DataFrame(
    outlier_summary,
    columns=['Column','IQR_Outliers','ZScore_Outliers']
)

outlier_df.to_csv('outlier_detection.csv', index=False)

outlier_df


In [ ]:

# Cell 7: Correlation Analysis

pearson_corr = df[numeric_cols].corr(method='pearson')
spearman_corr = df[numeric_cols].corr(method='spearman')

plt.figure(figsize=(10,8))
sns.heatmap(pearson_corr, cmap='coolwarm')
plt.title('Pearson Correlation')
plt.show()


In [ ]:

# Cell 8: Segment Analysis

if 'product_category' in df.columns and 'final_price' in df.columns:

    category_analysis = df.groupby('product_category').agg(
        Revenue=('final_price','sum'),
        AOV=('final_price','mean')
    )

    display(category_analysis)


In [ ]:

# Cell 9: Time Series Analysis

date_col = None

for col in df.columns:
    if 'date' in col.lower():
        date_col = col
        break

if date_col:

    df[date_col] = pd.to_datetime(df[date_col])

    daily = df.groupby(date_col)['final_price'].sum().reset_index()

    daily['MA7'] = daily['final_price'].rolling(7).mean()
    daily['MA14'] = daily['final_price'].rolling(14).mean()

    plt.figure(figsize=(12,5))
    plt.plot(daily[date_col], daily['final_price'])
    plt.plot(daily[date_col], daily['MA7'])
    plt.plot(daily[date_col], daily['MA14'])
    plt.show()


In [ ]:

# Cell 10: Logistic Regression

if 'customer_type' in df.columns:

    temp = df.copy()

    temp['target'] = np.where(
        temp['customer_type'].astype(str).str.lower().str.contains('return'),
        1,0
    )

    X = temp.select_dtypes(include=np.number).drop(
        columns=['target'],
        errors='ignore'
    )

    X = X.fillna(X.median())

    y = temp['target']

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X,y,test_size=0.2,random_state=42
    )

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train,y_train)

    preds = model.predict(X_test)

    print(classification_report(y_test,preds))
